In [ ]:
### Imports ###

import pandas as pd
import os
import glob

# Map the long ACA names to the short keys used.

dam_name_mapping = {
    "Embassament de Sau (Vilanova de Sau)": "sau",
    "Embassament de Susqueda (Osor)": "susqueda",
    "Embassament de la Baells (Cercs)": "baells",
    "Embassament de la Llosa del Cavall (Navès)": "llosa de cavall", 
    "Embassament de Sant Ponç (Clariana de Cardener)": "sant ponç", 
    "Embassament de Darnius Boadella (Darnius)": "boadella",
    "Embassament de Foix (Castellet i la Gornal)": "foix",
    "Embassament de Riudecanyes": "riudecanyes",
    "Embassament de Siurana (Cornudella de Montsant)": "siurana"
}

print("Loading ACA data...")

# Read and clean the volume data

df = pd.read_csv("../data/capacity_reservoirs.csv")
df['Volum embassat (hm3)'] = df['Volum embassat (hm3)'].str.replace(',', '.').astype(float)

df['Dia'] = pd.to_datetime(df['Dia'], format='%d/%m/%Y').dt.strftime('%Y-%m-%d')

df['short_name'] = df['Estació'].map(dam_name_mapping)

# Match ground truth volume labels and
# raster images

dataset_rows = []
tiff_files = glob.glob("../data/dataset_ndwi/*.tiff")
print(f"Found {len(tiff_files)} downloaded tensors. Matching with ground truth...")

matched_count = 0
missing_count = 0

for filepath in tiff_files:
    # Extract the filename
    filename = os.path.basename(filepath)
    
    # Split the name and date: "sau", "2023-04-15"
    name_part, date_part = filename.replace(".tiff", "").rsplit("_", 1)

    match = df[(df['short_name'] == name_part) & (df['Dia'] == date_part)]
    
    if not match.empty:
        # If match, grab the volume label
        pct = match['Volum embassat (hm3)'].values[0]
        dataset_rows.append({
            "filepath": filepath,
            "dam_name": name_part,
            "date": date_part,
            "volum": pct
        })
        matched_count += 1
    else:
        missing_count += 1

# Save the result

final_df = pd.DataFrame(dataset_rows)
final_df.to_csv("../data/master_labels_volume.csv", index=False)

print(f"\nDone! Successfully matched {matched_count} images.")
print(f"Saved to 'master_labels_volume.csv'.")
if missing_count > 0:
    print(f"Note: {missing_count} images couldn't find a matching date in the CSV.")

Loading ACA data...
Found 2284 downloaded tensors. Matching with ground truth...

Done! Successfully matched 2284 images.
Saved to 'master_labels_volume.csv'.
